In [13]:
import yfinance as yf
import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.api as sm

In [16]:
market = yf.download("^GSPC", start="2022-01-01", end="2024-01-01",auto_adjust=True)["Close"]
fund = yf.download("ARKK", start="2022-01-01", end="2024-01-01",auto_adjust=True)["Close"]
rf = yf.download("^IRX", start="2022-01-01", end="2024-01-01",auto_adjust=True)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [17]:
fund_ret = fund.pct_change().dropna()
mkt_ret = market.pct_change().dropna()

rf_daily = (rf["Close"] / 100) / 252

In [18]:
data = pd.concat([fund_ret, mkt_ret, rf_daily], axis=1).dropna()
data.columns = ["fund", "market", "rf"]
data.head()

,fund,market,rf
Date,,,
2022-01-04,-0.044334,-0.000630,0.000003
2022-01-05,-0.070881,-0.019393,0.000003
2022-01-06,-0.006270,-0.000964,0.000004
2022-01-07,-0.013555,-0.004050,0.000003
2022-01-10,0.002606,-0.001441,0.000004


In [19]:
# Simplify risk-free

data["fund_excess"] = data["fund"] - data["rf"]
data["market_excess"] = data["market"] - data["rf"]

In [20]:
data.head()

,fund,market,rf,fund_excess,market_excess
Date,,,,,
2022-01-04,-0.044334,-0.000630,0.000003,-0.044338,-0.000633
2022-01-05,-0.070881,-0.019393,0.000003,-0.070885,-0.019396
2022-01-06,-0.006270,-0.000964,0.000004,-0.006274,-0.000967
2022-01-07,-0.013555,-0.004050,0.000003,-0.013558,-0.004054
2022-01-10,0.002606,-0.001441,0.000004,0.002603,-0.001445


In [23]:
# CAPM Regression

X = sm.add_constant(data["market_excess"])
y = data["fund_excess"]

model = sm.OLS(y, X).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            fund_excess   R-squared:                       0.579
Model:                            OLS   Adj. R-squared:                  0.579
Method:                 Least Squares   F-statistic:                     685.9
Date:                Fri, 29 May 2026   Prob (F-statistic):           1.07e-95
Time:                        16:57:55   Log-Likelihood:                 1180.3
No. Observations:                 500   AIC:                            -2357.
Df Residuals:                     498   BIC:                            -2348.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -0.0006      0.001     -0.559      0.576      -0.003       0.001
market_excess     2.1857      0.083     26.189      0.000       2.022       2.350
==============================================================================
Omnibus:                        7.227   Durbin-Watson:                   1.988
Prob(Omnibus):                  0.027   Jarque-Bera (JB):                9.273
Skew:                           0.137   Prob(JB):                      0.00969
Kurtosis:                       3.608   Cond. No.                         81.6
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""